# 3.1 — Machine Learning Framing

Machine learning framing is the step where we decide what counts as data, what counts as a target, what loss the learner is allowed to optimize, and how we will choose among alternatives before touching a fancy algorithm. In this lesson, we build that contract from scratch with tiny NumPy examples: empirical risk, supervised versus unsupervised signals, semi- and self-supervised labels, validation, costs, and the final decision score.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build ML framing one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is spelled out so the modeling contract is never a black box. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, vectorized losses, and reproducible toy data.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any random demonstration.

### 1. Frame the rows, signal, target, and hypothesis family

A machine-learning problem begins by deciding what each row means. In a supervised frame, each row has an input vector `x` and a target `y`; the learner searches a hypothesis family for a rule that maps `x` to predictions. The same numeric table could be framed differently, so we write the contract explicitly before optimizing anything.

In [ ]:
X_w = np.array([[0.0, 1.0], [1.0, 1.0], [2.0, 1.0], [3.0, 1.0]])  # four rows; second column is an intercept feature.
y_w = np.array([0.2, 1.1, 1.9, 3.2])  # supervised targets for the four rows.
w_candidate_w = np.array([0.9, 0.1])  # one allowed linear rule: prediction = 0.9*x + 0.1.
pred_w = X_w @ w_candidate_w  # apply the rule to every row.
print("X shape:", X_w.shape, "y shape:", y_w.shape)  # inspect the supervised contract.
print("predictions:", np.round(pred_w, 3))  # inspect what the candidate hypothesis says.

▶ What you'll see: four input rows, four targets, and one prediction per row from a simple linear rule.

In [ ]:
plt.figure(figsize=(4.4, 3.2))  # create a compact supervised-framing plot.
plt.scatter(X_w[:, 0], y_w, color="black", label="target y")  # observed labels.
plt.plot(X_w[:, 0], pred_w, color="teal", marker="o", label="candidate f(x)")  # model output.
plt.title("1: supervised frame = inputs + targets + rule")
plt.xlabel("feature x"); plt.ylabel("target / prediction"); plt.legend(); plt.show()

▶ What you'll see: the candidate line follows the target points but does not match them perfectly.

*Why it's done this way:* the matrix `X` defines what information the learner may use, `y` defines what it must predict, and `w_candidate_w` defines one member of the allowed family. Writing those pieces separately prevents the common mistake of optimizing before the prediction task is even well specified.

### 2. Empirical risk: average the loss over examples

The core ML formula is empirical risk minimization:

$$\hat f=\arg\min_{f\in\mathcal F}\frac{1}{m}\sum_{i=1}^{m}\ell(f(x_i),y_i).$$

For squared loss, each example contributes $(\hat y_i-y_i)^2$. The average matters because it puts datasets of different sizes on a comparable scale; it is the training score the method is explicitly trying to reduce.

In [ ]:
losses_w = np.array([0.191, 0.122, 0.522])  # verified per-example losses from the lesson prose.
risk_w = float(np.mean(losses_w))  # empirical risk = average loss.
print("losses:", losses_w)  # inspect the individual terms in the sum.
print("empirical risk:", round(risk_w, 3))  # (0.191 + 0.122 + 0.522) / 3.
assert round(risk_w, 3) == 0.278  # self-check the lesson arithmetic.

▶ What you'll see: the three losses average to `0.278`.

In [ ]:
plt.figure(figsize=(4.4, 3.0))  # visualize each term in the empirical average.
plt.bar(["row 1", "row 2", "row 3"], losses_w, color="steelblue")  # one bar per example.
plt.axhline(risk_w, color="crimson", linestyle="--", label=f"mean={risk_w:.3f}")  # average loss.
plt.title("2: empirical risk is an average"); plt.ylabel("loss"); plt.legend(); plt.show()

▶ What you'll see: the mean line summarizes the three per-row penalties.

*Why it's done this way:* summing losses would make a larger dataset look worse just because it has more rows. Dividing by `m` turns total disagreement into the expected disagreement for one sampled example, which is the quantity we hope will transfer to future rows.

### 3. Choose the learning signal: supervised, unsupervised, semi-supervised, or self-supervised

The same raw examples can support different learning signals. Supervised learning uses provided `y`; unsupervised learning uses only `X` and searches for structure; semi-supervised learning mixes a few labeled rows with many unlabeled rows; self-supervised learning creates labels from the data itself. Framing is the decision about which signal is legitimate for the task.

In [ ]:
X_signal_w = np.array([[0.0, 0.2], [0.2, 0.1], [2.8, 3.0], [3.1, 2.9], [1.5, 1.6]])  # five rows in two clusters plus a middle row.
y_some_w = np.array([0.0, np.nan, 1.0, np.nan, np.nan])  # only two rows have human labels.
label_mask_w = ~np.isnan(y_some_w)  # semi-supervised mask: known labels versus unlabeled rows.
print("labeled rows:", np.where(label_mask_w)[0])  # inspect where target information exists.
print("unlabeled rows:", np.where(~label_mask_w)[0])  # inspect rows that still carry structure.

▶ What you'll see: only rows 0 and 2 are labeled, while the other rows can still shape the geometry.

In [ ]:
center0_w = X_signal_w[0]  # one labeled class-0 prototype.
center1_w = X_signal_w[2]  # one labeled class-1 prototype.
d0_w = np.linalg.norm(X_signal_w - center0_w, axis=1)  # distances to class-0 prototype.
d1_w = np.linalg.norm(X_signal_w - center1_w, axis=1)  # distances to class-1 prototype.
pseudo_w = (d1_w < d0_w).astype(float)  # self/semi-style pseudo-label from nearest prototype.
print("dist to class 0:", np.round(d0_w, 2))  # inspect geometry.
print("dist to class 1:", np.round(d1_w, 2))  # inspect geometry.
print("pseudo-labels:", pseudo_w)  # labels inferred from structure.

▶ What you'll see: unlabeled rows near a prototype inherit its pseudo-label.

In [ ]:
plt.figure(figsize=(4.2, 3.4))  # plot the signal choices.
plt.scatter(X_signal_w[:, 0], X_signal_w[:, 1], c=pseudo_w, cmap="coolwarm", s=90, edgecolor="k")  # color by inferred signal.
for i_w, xy_w in enumerate(X_signal_w):
    plt.text(xy_w[0] + 0.04, xy_w[1], f"r{i_w}")  # label rows.
plt.title("3: labels may be given or inferred"); plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.show()

▶ What you'll see: two visible groups, with unlabeled points assigned by proximity.

*Why it's done this way:* the loss can only optimize the signal we define. If labels are scarce, geometry can help, but pseudo-labels are weaker evidence than human labels because they are generated by an assumption — here, that nearby points should share a class.

### 4. Add the method's cost before selecting a model

The lesson's decision score is not just raw empirical risk. A complexity, regularization, or operational cost is added to avoid rewarding a flexible method merely for chasing the training table.

In [ ]:
raw_risk_w = 0.278  # empirical risk from the verified toy losses.
cost_w = 0.060  # complexity/regularization/operational cost from the lesson.
score_w = raw_risk_w + cost_w  # decision score = fit term + cost term.
print("raw risk:", raw_risk_w)  # inspect the fit component.
print("cost:", cost_w)  # inspect the guardrail component.
print("decision score:", round(score_w, 3))  # 0.338.
assert round(score_w, 3) == 0.338  # self-check the lesson arithmetic.

▶ What you'll see: the attractive raw score becomes `0.338` after the cost is included.

In [ ]:
plt.figure(figsize=(4.4, 3.0))  # decompose the score.
plt.bar(["raw risk", "cost", "total score"], [raw_risk_w, cost_w, score_w], color=["teal", "orange", "purple"])
plt.title("4: model selection uses the full score"); plt.ylabel("score"); plt.show()

▶ What you'll see: the total bar is the sum of the training-fit bar and the cost bar.

*Why it's done this way:* optimization pressure always finds shortcuts. The cost term encodes our prior belief that a more flexible, expensive, or brittle rule must earn its keep; mathematically it changes the argmin so the model is rewarded for durable fit, not just small training loss.

### 5. Validate the gap against a tempting alternative

A flexible alternative can look appealing, but the selection question is comparative. Here the alternative score is `0.378`, so the baseline-with-cost at `0.338` wins by an absolute gap of `0.040` and a relative gap of about `10.6%`.

In [ ]:
baseline_score_w = 0.338  # fit + cost for the simpler framed option.
flex_score_w = 0.378  # decision score for a tempting flexible alternative.
gap_w = flex_score_w - baseline_score_w  # absolute evidence for the simpler option.
rel_gap_w = gap_w / flex_score_w  # scale the gap by the alternative score.
print("gap:", round(gap_w, 3))  # 0.040.
print("relative gap:", round(rel_gap_w, 3))  # 0.106.
assert round(gap_w, 3) == 0.040
assert round(rel_gap_w, 3) == 0.106

▶ What you'll see: the simpler option is lower by `0.040`, about `10.6%` of the flexible score.

In [ ]:
plt.figure(figsize=(4.4, 3.0))  # compare candidate decision scores.
plt.bar(["baseline", "flexible"], [baseline_score_w, flex_score_w], color=["seagreen", "indianred"])
plt.ylabel("lower is better"); plt.title("5: compare full scores, not vibes"); plt.show()

▶ What you'll see: the baseline bar is lower, but the visible gap is modest.

*Why it's done this way:* a difference only matters if it is large enough to survive noise, resampling, or operational uncertainty. Reporting both absolute and relative gaps keeps us from declaring victory over a tiny, scale-dependent improvement.

### 6. Stabilization and the final framed decision

The lesson ends by turning a stability knob: a constrained version reduces the score by 20%, so `0.80 × 0.338 = 0.270`. The final decision compares baseline, flexible, and stabilized scores with one rule: lower full decision score wins.

In [ ]:
stable_score_w = 0.80 * baseline_score_w  # apply the stabilizing 20% reduction.
candidates_w = np.array([baseline_score_w, flex_score_w, stable_score_w])  # all framed options.
names_w = np.array(["baseline", "flexible", "stabilized"])  # readable labels.
best_idx_w = int(np.argmin(candidates_w))  # lower score wins.
print("scores:", dict(zip(names_w, np.round(candidates_w, 3))))  # inspect all choices.
print("winner:", names_w[best_idx_w], "score:", round(candidates_w[best_idx_w], 3))  # final decision.
assert round(stable_score_w, 3) == 0.270
assert names_w[best_idx_w] == "stabilized"

▶ What you'll see: the stabilized score is `0.270`, the minimum of the three candidates.

In [ ]:
plt.figure(figsize=(4.8, 3.0))  # visualize the final argmin.
colors_w = ["gray", "indianred", "seagreen"]  # green marks the winner.
plt.bar(names_w, candidates_w, color=colors_w)
plt.ylabel("decision score (lower is better)"); plt.title("6: framed model-selection decision"); plt.show()

▶ What you'll see: the stabilized option is the lowest bar and is therefore carried forward.

*Why it's done this way:* framing turns subjective modeling choices into a repeatable comparison. The final `argmin` is only meaningful because every candidate was scored on the same contract: the same risk scale, the same cost logic, and the same validation interpretation.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, losses, masks, and small vectorized ML framing demos.
import matplotlib.pyplot as plt  # load Matplotlib for line plots, scatters, bars, and score diagnostics.
np.random.seed(0)  # make all examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Name the inputs and target

**Goal.** Separate features from targets, because supervised ML begins with a contract about what is available at prediction time and what must be predicted. We build it in 2 steps.

In [ ]:
X_b1 = np.array([[0.0, 1.0], [1.0, 1.0], [2.0, 1.0], [3.0, 1.0]])  # feature plus intercept column.
y_b1 = np.array([0.2, 1.1, 1.9, 3.2])  # supervised target values.
print("features shape:", X_b1.shape)  # inspect rows and columns.
print("target shape:", y_b1.shape)  # inspect one target per row.
assert X_b1.shape == (4, 2) and y_b1.shape == (4,)

▶ What you'll see: four examples, two feature columns, and four target values.

In [ ]:
plt.figure(figsize=(4, 3))  # create a compact target plot.
plt.scatter(X_b1[:, 0], y_b1, color="black")  # show target against the real-valued feature.
plt.title("Basic 1: targets attached to rows"); plt.xlabel("feature"); plt.ylabel("target y"); plt.show()

▶ What you'll see: each row has exactly one target point.

👀 Takeaway: supervised framing is impossible until inputs and targets are explicitly paired.

### Basic 2 — Compute one model's predictions

**Goal.** Apply a candidate rule to every row, because losses compare predictions with targets, not raw parameters with targets. We build it in 2 steps.

In [ ]:
X_b2 = np.array([[0.0, 1.0], [1.0, 1.0], [2.0, 1.0], [3.0, 1.0]])  # same feature table.
w_b2 = np.array([0.9, 0.1])  # slope and intercept for one linear hypothesis.
pred_b2 = X_b2 @ w_b2  # vectorized prediction for all rows.
print("predictions:", np.round(pred_b2, 3))  # inspect f(x_i).
assert np.allclose(pred_b2, [0.1, 1.0, 1.9, 2.8])

▶ What you'll see: the candidate rule outputs one prediction for each input row.

In [ ]:
plt.figure(figsize=(4, 3))  # create a line plot of the rule.
plt.plot(X_b2[:, 0], pred_b2, marker="o", color="teal")  # visualize f(x).
plt.title("Basic 2: candidate hypothesis outputs"); plt.xlabel("feature"); plt.ylabel("prediction"); plt.show()

▶ What you'll see: a straight-line prediction rule across the four examples.

👀 Takeaway: a hypothesis is a function that turns available inputs into predicted outputs.

### Basic 3 — Turn errors into losses

**Goal.** Convert prediction misses into squared losses, because the loss defines what the learner is punished for. We build it in 3 steps.

In [ ]:
pred_b3 = np.array([0.1, 1.0, 1.9, 2.8])  # candidate predictions.
y_b3 = np.array([0.2, 1.1, 1.9, 3.2])  # targets.
err_b3 = pred_b3 - y_b3  # signed residuals.
print("errors:", np.round(err_b3, 3))  # inspect over- and under-predictions.

▶ What you'll see: two small negative misses, one exact hit, and one larger miss.

In [ ]:
loss_b3 = err_b3 ** 2  # squared loss per example.
print("squared losses:", np.round(loss_b3, 3))  # inspect nonnegative penalties.
assert np.allclose(loss_b3, [0.01, 0.01, 0.0, 0.16])

▶ What you'll see: squaring removes signs and makes the larger miss cost much more.

In [ ]:
plt.figure(figsize=(4, 3))  # visualize per-row penalties.
plt.bar(["r0", "r1", "r2", "r3"], loss_b3, color="orange")  # one loss per row.
plt.title("Basic 3: squared loss by row"); plt.ylabel("loss"); plt.show()

▶ What you'll see: row 3 dominates the training penalty.

👀 Takeaway: loss is the mathematical definition of what counts as a mistake.

### Basic 4 — Average losses into empirical risk

**Goal.** Compute the verified lesson average, because ERM minimizes the mean loss over training examples. We build it in 2 steps.

In [ ]:
losses_b4 = np.array([0.191, 0.122, 0.522])  # verified lesson losses.
sum_b4 = float(np.sum(losses_b4))  # numerator of empirical risk.
print("sum of losses:", round(sum_b4, 3))  # inspect 0.835.
assert round(sum_b4, 3) == 0.835

▶ What you'll see: the three example losses sum to `0.835`.

In [ ]:
risk_b4 = float(np.mean(losses_b4))  # average over m=3 examples.
print("empirical risk:", round(risk_b4, 3))  # inspect 0.278.
assert round(risk_b4, 3) == 0.278
plt.figure(figsize=(4, 3))
plt.bar(["sum", "mean"], [sum_b4, risk_b4], color=["gray", "steelblue"])
plt.title("Basic 4: sum versus average"); plt.ylabel("loss scale"); plt.show()

▶ What you'll see: the mean is the size-normalized training score.

👀 Takeaway: empirical risk is the average loss, not the raw total loss.

### Basic 5 — Add a complexity cost

**Goal.** Add the lesson's cost term to the empirical risk, because the chosen score must include the method's guardrail. We build it in 2 steps.

In [ ]:
risk_b5 = 0.278  # verified empirical risk.
cost_b5 = 0.060  # complexity, regularization, or operational cost.
score_b5 = risk_b5 + cost_b5  # full decision score.
print("score:", round(score_b5, 3))  # inspect risk + cost.
assert round(score_b5, 3) == 0.338

▶ What you'll see: the full score is `0.338`.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["risk", "cost", "risk+cost"], [risk_b5, cost_b5, score_b5], color=["teal", "orange", "purple"])
plt.title("Basic 5: full model-selection score"); plt.ylabel("score"); plt.show()

▶ What you'll see: the decision score is larger than the raw fit term.

👀 Takeaway: model selection should compare the complete objective, not only the training loss.

### Basic 6 — Compare two framed options

**Goal.** Compare a baseline with a flexible alternative on the same score scale, because lower numbers only mean better when the units match. We build it in 2 steps.

In [ ]:
scores_b6 = np.array([0.338, 0.378])  # baseline and flexible alternative scores.
names_b6 = np.array(["baseline", "flexible"])  # readable labels.
winner_b6 = int(np.argmin(scores_b6))  # lower score wins.
print("winner:", names_b6[winner_b6])  # inspect selected option.
assert names_b6[winner_b6] == "baseline"

▶ What you'll see: the baseline wins because `0.338 < 0.378`.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(names_b6, scores_b6, color=["seagreen", "indianred"])
plt.title("Basic 6: lower comparable score wins"); plt.ylabel("decision score"); plt.show()

▶ What you'll see: the baseline bar is lower than the flexible bar.

👀 Takeaway: comparisons are meaningful only after every option is evaluated on the same scale.

### Basic 7 — Measure the validation gap

**Goal.** Compute absolute and relative gaps, because a tiny score difference may not be stable enough to trust. We build it in 2 steps.

In [ ]:
baseline_b7 = 0.338  # lower-scoring option.
flex_b7 = 0.378  # competing option.
gap_b7 = flex_b7 - baseline_b7  # absolute gap.
print("absolute gap:", round(gap_b7, 3))  # inspect 0.040.
assert round(gap_b7, 3) == 0.040

▶ What you'll see: the two options differ by `0.040`.

In [ ]:
rel_b7 = gap_b7 / flex_b7  # relative gap against the larger score.
print("relative gap:", round(rel_b7, 3))  # inspect about 0.106.
assert round(rel_b7, 3) == 0.106
plt.figure(figsize=(4, 3))
plt.bar(["absolute", "relative"], [gap_b7, rel_b7], color=["navy", "gray"])
plt.title("Basic 7: gap diagnostics"); plt.ylabel("gap value"); plt.show()

▶ What you'll see: the relative gap reports the win on the score's own scale.

👀 Takeaway: gaps are evidence strength, not decorative arithmetic.

### Basic 8 — Apply a stabilization knob

**Goal.** Reduce the baseline score by 20%, because constraints or regularization can intentionally trade flexibility for stability. We build it in 2 steps.

In [ ]:
base_b8 = 0.338  # baseline full score.
multiplier_b8 = 0.80  # 20% reduction means keeping 80%.
stable_b8 = multiplier_b8 * base_b8  # stabilized score.
print("stabilized score:", round(stable_b8, 3))  # inspect 0.270.
assert round(stable_b8, 3) == 0.270

▶ What you'll see: stabilization changes `0.338` into `0.270`.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["before", "after"], [base_b8, stable_b8], color=["gray", "seagreen"])
plt.title("Basic 8: stability knob lowers score"); plt.ylabel("decision score"); plt.show()

▶ What you'll see: the after bar is 20% lower than the before bar.

👀 Takeaway: a stabilizing constraint belongs in the framing if it changes future decision quality.

### Basic 9 — Pick the final minimum

**Goal.** Use `argmin` over all candidate scores, because framing ends with a repeatable selection rule. We build it in 2 steps.

In [ ]:
scores_b9 = np.array([0.338, 0.378, 0.270])  # baseline, flexible, stabilized.
labels_b9 = np.array(["baseline", "flexible", "stabilized"])  # candidate names.
best_b9 = int(np.argmin(scores_b9))  # locate the lowest score.
print("best option:", labels_b9[best_b9], "score:", scores_b9[best_b9])  # inspect final selection.
assert labels_b9[best_b9] == "stabilized"

▶ What you'll see: the stabilized option is selected.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(labels_b9, scores_b9, color=["gray", "indianred", "seagreen"])
plt.title("Basic 9: final framed decision"); plt.ylabel("lower is better"); plt.show()

▶ What you'll see: the selected option is the lowest bar.

👀 Takeaway: the final model choice is an optimization over the framed decision score.

### Basic 10 — Distinguish labeled and unlabeled rows

**Goal.** Build a label mask, because semi-supervised framing uses some human labels while still retaining unlabeled examples. We build it in 2 steps.

In [ ]:
y_b10 = np.array([0.0, np.nan, 1.0, np.nan, np.nan])  # two labels and three unlabeled rows.
labeled_b10 = ~np.isnan(y_b10)  # True where a target is available.
print("labeled mask:", labeled_b10.astype(int))  # inspect supervised evidence.
assert int(np.sum(labeled_b10)) == 2

▶ What you'll see: only two rows have direct target labels.

In [ ]:
counts_b10 = np.array([np.sum(labeled_b10), np.sum(~labeled_b10)])  # labeled versus unlabeled counts.
plt.figure(figsize=(4, 3))
plt.bar(["labeled", "unlabeled"], counts_b10, color=["teal", "orange"])
plt.title("Basic 10: semi-supervised data mix"); plt.ylabel("row count"); plt.show()

▶ What you'll see: unlabeled rows outnumber labeled rows in this toy semi-supervised frame.

👀 Takeaway: the learning signal is part of the frame; unlabeled rows are not the same as negative labels.

## 🟡 Easy

### Easy 1 — Evaluate two losses for the same predictions

**Goal.** Compare squared loss with absolute loss, because different task framings punish mistakes differently. We build it in 3 steps.

In [ ]:
pred_e1 = np.array([0.1, 1.0, 1.9, 2.8])  # candidate predictions.
y_e1 = np.array([0.2, 1.1, 1.9, 3.2])  # targets.
err_e1 = pred_e1 - y_e1  # residuals.
print("errors:", np.round(err_e1, 3))  # inspect signed misses.

▶ What you'll see: the same residuals will feed two different losses.

In [ ]:
mse_e1 = float(np.mean(err_e1 ** 2))  # mean squared error.
mae_e1 = float(np.mean(np.abs(err_e1)))  # mean absolute error.
print("MSE:", round(mse_e1, 3), "MAE:", round(mae_e1, 3))  # compare scales.
assert round(mse_e1, 3) == 0.045 and round(mae_e1, 3) == 0.150

▶ What you'll see: squared loss reports `0.045`, while absolute loss reports `0.150`.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["MSE", "MAE"], [mse_e1, mae_e1], color=["purple", "steelblue"])
plt.title("Easy 1: loss choice changes the score"); plt.ylabel("average loss"); plt.show()

▶ What you'll see: the two bars are on different scales even for identical predictions.

👀 Takeaway: pick the loss before comparing models, because the loss defines the target behavior.

### Easy 2 — Split train and validation rows

**Goal.** Compute training and validation risk separately, because the training number alone cannot tell whether the rule will generalize. We build it in 3 steps.

In [ ]:
X_e2 = np.arange(6, dtype=float)  # six ordered examples.
y_e2 = np.array([0.1, 0.8, 2.2, 2.9, 4.4, 5.1])  # targets.
pred_e2 = 0.9 * X_e2 + 0.2  # one candidate rule.
train_mask_e2 = np.array([True, True, True, True, False, False])  # first four rows for fitting.
print("train rows:", np.where(train_mask_e2)[0], "validation rows:", np.where(~train_mask_e2)[0])  # inspect split.

▶ What you'll see: rows 0–3 are train, rows 4–5 are validation.

In [ ]:
loss_e2 = (pred_e2 - y_e2) ** 2  # squared loss for every row.
train_risk_e2 = float(np.mean(loss_e2[train_mask_e2]))  # average training loss.
val_risk_e2 = float(np.mean(loss_e2[~train_mask_e2]))  # average validation loss.
print("train risk:", round(train_risk_e2, 3), "validation risk:", round(val_risk_e2, 3))  # compare future proxy.
assert round(train_risk_e2, 3) == 0.035 and round(val_risk_e2, 3) == 0.260

▶ What you'll see: validation loss is higher than training loss.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["train", "validation"], [train_risk_e2, val_risk_e2], color=["teal", "orange"])
plt.title("Easy 2: train vs validation risk"); plt.ylabel("MSE"); plt.show()

▶ What you'll see: the validation bar warns that the apparent fit weakened on held-out rows.

👀 Takeaway: validation is the notebook's first proxy for future behavior.

### Easy 3 — Build pseudo-labels from unlabeled data

**Goal.** Assign unlabeled rows by nearest labeled prototype, because semi-supervised framing often turns structure into provisional targets. We build it in 3 steps.

In [ ]:
X_e3 = np.array([[0.0, 0.2], [0.2, 0.1], [2.8, 3.0], [3.1, 2.9], [1.5, 1.6]])  # toy geometry.
y_e3 = np.array([0.0, np.nan, 1.0, np.nan, np.nan])  # two labeled prototypes.
known_e3 = ~np.isnan(y_e3)  # label availability mask.
print("known label rows:", np.where(known_e3)[0])  # inspect prototypes.

▶ What you'll see: row 0 and row 2 anchor the two classes.

In [ ]:
proto0_e3 = X_e3[0]  # class-0 prototype.
proto1_e3 = X_e3[2]  # class-1 prototype.
d0_e3 = np.linalg.norm(X_e3 - proto0_e3, axis=1)  # distance to class 0.
d1_e3 = np.linalg.norm(X_e3 - proto1_e3, axis=1)  # distance to class 1.
pseudo_e3 = (d1_e3 < d0_e3).astype(float)  # nearest-prototype pseudo-label.
print("pseudo-labels:", pseudo_e3)  # inspect inferred labels.
assert np.allclose(pseudo_e3, [0, 0, 1, 1, 1])

▶ What you'll see: rows near the high-valued prototype receive label 1.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(X_e3[:, 0], X_e3[:, 1], c=pseudo_e3, cmap="coolwarm", s=90, edgecolor="k")
plt.title("Easy 3: nearest-prototype pseudo-labels"); plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.show()

▶ What you'll see: pseudo-label colors follow the two visible regions.

👀 Takeaway: pseudo-labels are useful only because the frame assumes nearby rows should behave similarly.

### Easy 4 — Choose a threshold from decision loss

**Goal.** Compare classification thresholds with a cost-sensitive loss, because framing often includes the business cost of false decisions. We build it in 3 steps.

In [ ]:
probs_e4 = np.array([0.10, 0.35, 0.55, 0.80, 0.90])  # predicted positive probabilities.
y_e4 = np.array([0, 0, 1, 1, 1])  # true binary labels.
thresholds_e4 = np.array([0.30, 0.50, 0.70])  # candidate decision thresholds.
print("thresholds:", thresholds_e4)  # inspect choices.

▶ What you'll see: three possible rules for turning probabilities into actions.

In [ ]:
costs_e4 = []  # store total cost per threshold.
for t_e4 in thresholds_e4:
    pred_class_e4 = (probs_e4 >= t_e4).astype(int)  # threshold probabilities.
    fp_e4 = int(np.sum((pred_class_e4 == 1) & (y_e4 == 0)))  # false positives.
    fn_e4 = int(np.sum((pred_class_e4 == 0) & (y_e4 == 1)))  # false negatives.
    costs_e4.append(fp_e4 * 1 + fn_e4 * 3)  # false negatives cost three times more.
print("costs:", costs_e4)  # inspect cost-sensitive score.
assert costs_e4 == [1, 0, 3]

▶ What you'll see: threshold `0.50` has the lowest cost in this toy setting.

In [ ]:
best_e4 = int(np.argmin(costs_e4))  # choose minimum cost threshold.
plt.figure(figsize=(4, 3))
plt.bar([str(t) for t in thresholds_e4], costs_e4, color="slateblue")
plt.title("Easy 4: threshold chosen by loss"); plt.xlabel("threshold"); plt.ylabel("decision cost"); plt.show()
print("best threshold:", thresholds_e4[best_e4])  # inspect selected threshold.

▶ What you'll see: the middle threshold is the lowest bar.

👀 Takeaway: a classifier's threshold is part of the ML frame when mistakes have unequal costs.

### Easy 5 — Rank options by full score

**Goal.** Combine risk, cost, and stability into one ranking, because model selection should be reproducible. We build it in 3 steps.

In [ ]:
names_e5 = np.array(["baseline", "flexible", "stabilized"])  # candidate model frames.
risk_e5 = np.array([0.278, 0.318, 0.216])  # raw fit terms.
cost_e5 = np.array([0.060, 0.060, 0.054])  # method costs.
print("raw risk:", risk_e5)  # inspect first component.
print("cost:", cost_e5)  # inspect second component.

▶ What you'll see: each candidate has a risk and a cost component.

In [ ]:
score_e5 = risk_e5 + cost_e5  # full comparable score.
order_e5 = np.argsort(score_e5)  # sort from best to worst.
print("scores:", dict(zip(names_e5, np.round(score_e5, 3))))  # inspect full scores.
print("ranking:", names_e5[order_e5])  # inspect ordered candidates.
assert np.allclose(np.round(score_e5, 3), [0.338, 0.378, 0.270])

▶ What you'll see: the full scores reproduce the lesson's `0.338`, `0.378`, and `0.270`.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(names_e5, score_e5, color=["gray", "indianred", "seagreen"])
plt.title("Easy 5: model ranking by full score"); plt.ylabel("lower is better"); plt.show()

▶ What you'll see: the stabilized model ranks first because it has the smallest full score.

👀 Takeaway: once the score is defined, selection is sorting, not storytelling.

## 🔴 Advanced

### Advanced 1 — Show overfitting with polynomial capacity

**Goal.** Compare train and validation MSE as polynomial degree increases, because a more flexible family can lower training loss while hurting future performance. We build it in 4 steps.

In [ ]:
x_a1 = np.linspace(-1, 1, 9)  # small one-dimensional dataset.
y_a1 = 1 + 2 * x_a1 + 0.25 * np.array([0, -1, 1, 0, 1, -1, 0, 1, 0])  # noisy trend.
train_a1 = np.array([True, True, True, True, True, False, False, False, False])  # first half train.
degrees_a1 = np.array([1, 2, 5])  # capacity choices.
print("degrees:", degrees_a1)  # inspect hypothesis-family sizes.

▶ What you'll see: the model family grows from a line to a degree-5 polynomial.

In [ ]:
train_mse_a1 = []  # store training errors.
val_mse_a1 = []  # store validation errors.
for d_a1 in degrees_a1:
    coef_a1 = np.polyfit(x_a1[train_a1], y_a1[train_a1], d_a1)  # fit polynomial on training rows only.
    pred_a1 = np.polyval(coef_a1, x_a1)  # predict all rows.
    train_mse_a1.append(float(np.mean((pred_a1[train_a1] - y_a1[train_a1]) ** 2)))  # train risk.
    val_mse_a1.append(float(np.mean((pred_a1[~train_a1] - y_a1[~train_a1]) ** 2)))  # validation risk.
print("train MSE:", np.round(train_mse_a1, 3))  # inspect fit.
print("validation MSE:", np.round(val_mse_a1, 3))  # inspect future proxy.
assert train_mse_a1[-1] <= train_mse_a1[0]

▶ What you'll see: higher degree lowers training error but can raise validation error.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(degrees_a1, train_mse_a1, marker="o", label="train")
plt.plot(degrees_a1, val_mse_a1, marker="s", label="validation")
plt.title("Advanced 1: capacity can overfit"); plt.xlabel("polynomial degree"); plt.ylabel("MSE"); plt.legend(); plt.show()

▶ What you'll see: train and validation curves do not necessarily move together.

In [ ]:
best_degree_a1 = int(degrees_a1[np.argmin(val_mse_a1)])  # choose by validation risk.
print("best validation degree:", best_degree_a1)  # inspect selected capacity.

▶ What you'll see: the chosen degree is based on validation, not training.

👀 Takeaway: hypothesis-family flexibility is part of the frame and must be checked against held-out data.

### Advanced 2 — Add a regularization penalty to model selection

**Goal.** Penalize large coefficients, because regularization charges a model for using fragile parameter magnitude. We build it in 4 steps.

In [ ]:
x_a2 = np.linspace(-1, 1, 9)  # toy feature.
y_a2 = 1 + 2 * x_a2 + 0.25 * np.array([0, -1, 1, 0, 1, -1, 0, 1, 0])  # noisy linear trend.
degrees_a2 = np.array([1, 2, 5])  # candidate families.
lam_a2 = 0.01  # coefficient-size penalty strength.
print("lambda:", lam_a2)  # inspect regularization strength.

▶ What you'll see: one penalty strength will be applied consistently across candidates.

In [ ]:
risk_a2 = []  # store training MSE.
penalty_a2 = []  # store coefficient penalty.
for d_a2 in degrees_a2:
    coef_a2 = np.polyfit(x_a2, y_a2, d_a2)  # fit candidate on all available training data for this demo.
    pred_a2 = np.polyval(coef_a2, x_a2)  # predictions.
    risk_a2.append(float(np.mean((pred_a2 - y_a2) ** 2)))  # empirical risk.
    penalty_a2.append(float(lam_a2 * np.sum(coef_a2 ** 2)))  # regularization cost.
print("risk:", np.round(risk_a2, 4))  # inspect fit component.
print("penalty:", np.round(penalty_a2, 4))  # inspect complexity component.

▶ What you'll see: higher-degree models often buy lower risk with larger penalties.

In [ ]:
score_a2 = np.array(risk_a2) + np.array(penalty_a2)  # regularized selection score.
best_a2 = int(np.argmin(score_a2))  # choose lowest full score.
print("regularized scores:", np.round(score_a2, 4))  # inspect final criterion.
print("chosen degree:", degrees_a2[best_a2])  # inspect winner.
assert score_a2[best_a2] == np.min(score_a2)

▶ What you'll see: the winner minimizes risk plus coefficient cost.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar([str(d) for d in degrees_a2], score_a2, color="purple")
plt.title("Advanced 2: regularized model score"); plt.xlabel("degree"); plt.ylabel("risk + λ||w||²"); plt.show()

▶ What you'll see: the lowest bar is the regularized selection.

👀 Takeaway: regularization changes the objective, so it changes which model the frame prefers.

### Advanced 3 — Bootstrap the score gap

**Goal.** Resample per-example losses to estimate gap uncertainty, because a small average gap can disappear under sampling noise. We build it in 4 steps.

In [ ]:
loss_base_a3 = np.array([0.20, 0.18, 0.31, 0.25, 0.16, 0.22])  # baseline validation losses.
loss_flex_a3 = np.array([0.23, 0.16, 0.35, 0.27, 0.18, 0.24])  # flexible validation losses.
observed_gap_a3 = float(np.mean(loss_flex_a3) - np.mean(loss_base_a3))  # positive means baseline lower.
print("observed gap:", round(observed_gap_a3, 3))  # inspect raw evidence.
assert round(observed_gap_a3, 3) == 0.018

▶ What you'll see: the baseline wins by about `0.018` validation loss.

In [ ]:
rng_a3 = np.random.default_rng(0)  # reproducible bootstrap.
gaps_a3 = []  # store resampled gaps.
for _a3 in range(500):
    idx_a3 = rng_a3.integers(0, len(loss_base_a3), len(loss_base_a3))  # resample rows with replacement.
    gaps_a3.append(float(np.mean(loss_flex_a3[idx_a3]) - np.mean(loss_base_a3[idx_a3])))  # resampled gap.
gaps_a3 = np.array(gaps_a3)  # convert to array for summaries.
print("bootstrap gap mean:", round(float(np.mean(gaps_a3)), 3))  # inspect typical gap.

▶ What you'll see: resampled gaps fluctuate around the observed gap.

In [ ]:
lo_a3, hi_a3 = np.percentile(gaps_a3, [5, 95])  # rough uncertainty interval.
print("5%-95% interval:", round(float(lo_a3), 3), round(float(hi_a3), 3))  # inspect uncertainty.
assert lo_a3 < observed_gap_a3 < hi_a3

▶ What you'll see: the interval shows how much the gap moves under resampling.

In [ ]:
plt.figure(figsize=(5, 3))
plt.hist(gaps_a3, bins=20, color="steelblue", edgecolor="white")
plt.axvline(0, color="black", linestyle="--", label="no gap")
plt.axvline(observed_gap_a3, color="crimson", label="observed")
plt.title("Advanced 3: bootstrap score-gap uncertainty"); plt.xlabel("flexible mean loss - baseline mean loss"); plt.legend(); plt.show()

▶ What you'll see: if much of the histogram is near zero, the apparent win is fragile.

👀 Takeaway: validation gaps should be interpreted with uncertainty, especially when the difference is small.

### Advanced 4 — Compare supervised and unsupervised objectives

**Goal.** Score the same rows with a label loss and a clustering distortion, because supervised and unsupervised frames optimize different quantities. We build it in 4 steps.

In [ ]:
X_a4 = np.array([[0.0, 0.1], [0.2, 0.0], [2.8, 3.0], [3.1, 2.9]])  # two visible clusters.
y_a4 = np.array([0, 0, 1, 1])  # supervised labels.
centers_a4 = np.array([[0.1, 0.05], [2.95, 2.95]])  # unsupervised cluster centers.
print("rows:", X_a4.shape[0], "centers:", centers_a4.shape[0])  # inspect dimensions.

▶ What you'll see: four rows and two cluster centers.

In [ ]:
supervised_pred_a4 = np.array([0, 0, 1, 1])  # label predictions from a classifier.
zero_one_a4 = float(np.mean(supervised_pred_a4 != y_a4))  # supervised classification loss.
print("supervised 0-1 loss:", zero_one_a4)  # inspect label objective.
assert zero_one_a4 == 0.0

▶ What you'll see: the supervised rule gets every provided label correct.

In [ ]:
dists_a4 = np.stack([np.sum((X_a4 - c_a4) ** 2, axis=1) for c_a4 in centers_a4], axis=1)  # squared distance to each center.
assign_a4 = np.argmin(dists_a4, axis=1)  # nearest cluster center.
distortion_a4 = float(np.mean(np.min(dists_a4, axis=1)))  # unsupervised objective.
print("cluster assignments:", assign_a4)  # inspect unsupervised structure.
print("distortion:", round(distortion_a4, 3))  # inspect clustering score.
assert round(distortion_a4, 3) == 0.019

▶ What you'll see: clustering also separates the rows, but it reports distance distortion instead of label error.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(X_a4[:, 0], X_a4[:, 1], c=assign_a4, cmap="coolwarm", s=90, edgecolor="k")
plt.scatter(centers_a4[:, 0], centers_a4[:, 1], marker="x", s=120, color="black")
plt.title("Advanced 4: unsupervised distortion"); plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.show()

▶ What you'll see: centers summarize geometry, not external class labels.

👀 Takeaway: supervised and unsupervised scores answer different questions, so they should not be ranked as if they were the same loss.

### Advanced 5 — Simulate self-supervised pretext labels

**Goal.** Create a label from the input itself and test whether it helps a downstream target, because self-supervised learning uses structure in `X` before scarce labels are available. We build it in 4 steps.

In [ ]:
X_a5 = np.array([[0.0, 0.2], [0.5, 0.4], [1.0, 1.1], [1.5, 1.4], [2.0, 2.2], [2.5, 2.4]])  # unlabeled sequence-like rows.
pretext_a5 = (X_a5[:, 1] > X_a5[:, 0]).astype(float)  # self-supervised label: is second coordinate larger?
print("pretext labels:", pretext_a5)  # inspect labels made from X.
assert int(np.sum(pretext_a5)) == 3

▶ What you'll see: every row has a generated label without external annotation.

In [ ]:
feature_a5 = (X_a5[:, 1] - X_a5[:, 0])[:, None]  # learned-style representation from the pretext relation.
downstream_y_a5 = np.array([0.1, 0.0, 0.2, -0.1, 0.3, 0.1])  # tiny downstream target.
w_a5 = np.linalg.pinv(feature_a5) @ downstream_y_a5  # fit a one-feature linear downstream rule.
pred_a5 = feature_a5 @ w_a5  # downstream predictions.
rmse_a5 = float(np.sqrt(np.mean((pred_a5 - downstream_y_a5) ** 2)))  # downstream fit quality.
print("representation weight:", np.round(w_a5, 3))  # inspect fitted rule.
print("downstream RMSE:", round(rmse_a5, 3))  # inspect utility.

▶ What you'll see: a representation derived from input structure can be reused in a labeled task.

In [ ]:
baseline_a5 = np.full_like(downstream_y_a5, np.mean(downstream_y_a5))  # no-representation baseline.
rmse_base_a5 = float(np.sqrt(np.mean((baseline_a5 - downstream_y_a5) ** 2)))  # baseline error.
print("baseline RMSE:", round(rmse_base_a5, 3))  # compare simple baseline.
assert rmse_a5 <= rmse_base_a5

▶ What you'll see: the pretext-derived feature is at least as good as the mean baseline in this toy case.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["pretext feature", "mean baseline"], [rmse_a5, rmse_base_a5], color=["teal", "gray"])
plt.title("Advanced 5: self-supervised representation check"); plt.ylabel("downstream RMSE"); plt.show()

▶ What you'll see: the lower bar indicates which framing gives the better downstream score.

👀 Takeaway: self-supervised learning is justified only if the generated task creates representations that help the real downstream decision.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Risk view: choose the data signal, target, and loss before choosing the algorithm.

ML framing (supervised/unsupervised/semi/self) uses empirical risk, validation behavior, and a cost-aware decision score. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine, load_breast_cancer, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier

np.random.seed(7)

def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []
    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))
    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))
    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))
    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))
    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))
    return rungs

def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)

def logistic_baseline(x_tr, y_tr, x_te):
    """Default classifier used to demonstrate a ladder end to end."""
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)


## The concept, built once on D1

The lesson formula is $$ \hat f=\arg\min_{f\in\mathcal F}\frac1m\sum_{i=1}^m\ell(f(x_i),y_i) $$. The next cell recomputes the exact loss average, cost, gap, and stabilized score from the plan.

In [ ]:

def ml_framing_supervised_unsupervised_semi_se_method():
    losses = np.array([0.191, 0.122, 0.522], dtype=float)
    raw_sum = float(losses.sum())
    empirical_risk = round(float(raw_sum / len(losses)), 3)
    cost = 0.060
    score = round(empirical_risk + cost, 3)
    alternative = 0.378
    gap = round(alternative - score, 3)
    relative_gap = round(gap / alternative, 3)
    stable_score = round(0.80 * score, 3)
    final_score = min(score, alternative, stable_score)
    return {
        "losses": losses,
        "sum": raw_sum,
        "risk": empirical_risk,
        "cost": cost,
        "score": score,
        "alternative": alternative,
        "gap": gap,
        "relative_gap": relative_gap,
        "stable": stable_score,
        "final": final_score,
    }

lesson_check = ml_framing_supervised_unsupervised_semi_se_method()
print("losses:", lesson_check["losses"])
print("R_S =", round(lesson_check["sum"], 3), "/ 3 =", round(lesson_check["risk"], 3))
print("score =", round(lesson_check["score"], 3))
print("gap =", round(lesson_check["gap"], 3))
print("relative gap =", round(lesson_check["relative_gap"], 3))
print("stable score =", round(lesson_check["stable"], 3))
assert np.isclose(round(lesson_check["sum"], 3), 0.835)
assert np.isclose(round(lesson_check["risk"], 3), 0.278)
assert np.isclose(round(lesson_check["score"], 3), 0.338)
assert np.isclose(round(lesson_check["gap"], 3), 0.040)
assert np.isclose(round(lesson_check["relative_gap"], 3), 0.106)
assert np.isclose(round(lesson_check["stable"], 3), 0.270)


The assertions above keep the notebook and lesson prose on the same algorithmic scale.

In [ ]:

def safe_stratify(y):
    values, counts = np.unique(y, return_counts=True)
    if len(values) < 2:
        return None
    if counts.min() < 2:
        return None
    return y

def plot_2d_projection(ax, X, y, title):
    x_plot = X[:, :2]
    ax.scatter(x_plot[:, 0], x_plot[:, 1], c=y, cmap="viridis", s=16, alpha=0.75)
    ax.set_title(title, fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])

def logistic_candidates_for_rung(X, y):
    stratify = safe_stratify(y)
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=stratify)
    scaler = StandardScaler()
    x_tr_s = scaler.fit_transform(x_tr)
    x_te_s = scaler.transform(x_te)
    candidates = []
    for c_value in [0.05, 0.2, 1.0, 5.0]:
        model = LogisticRegression(C=c_value, max_iter=2000)
        model.fit(x_tr_s, y_tr)
        tr_prob = model.predict_proba(x_tr_s)
        te_prob = model.predict_proba(x_te_s)
        tr_pred = model.predict(x_tr_s)
        te_pred = model.predict(x_te_s)
        labels = model.classes_
        train_loss = log_loss(y_tr, tr_prob, labels=labels)
        val_loss = log_loss(y_te, te_prob, labels=labels)
        cost = 0.02 / c_value
        candidates.append({
            "C": c_value,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "gap": float(val_loss - train_loss),
            "accuracy": float(accuracy_score(y_te, te_pred)),
            "cost": float(cost),
            "score": float(val_loss + cost),
            "pred": te_pred,
        })
    raw_winner = min(candidates, key=lambda item: item["val_loss"])
    fixed_winner = min(candidates, key=lambda item: item["score"])
    return candidates, raw_winner, fixed_winner

def run_logistic_ladder():
    rows = []
    for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
        candidates, raw_winner, fixed_winner = logistic_candidates_for_rung(X, y)
        rows.append({
            "rung": rung,
            "name": name,
            "n": X.shape[0],
            "d": X.shape[1],
            "classes": len(np.unique(y)),
            "metric": fixed_winner["val_loss"],
            "gap": fixed_winner["gap"],
            "accuracy": fixed_winner["accuracy"],
            "C": fixed_winner["C"],
            "score": fixed_winner["score"],
            "raw_C": raw_winner["C"],
        })
    return rows

def bias_variance_candidates_for_rung(X, y):
    stratify = safe_stratify(y)
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=stratify)
    models = [
        ("linear", make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))),
        ("flexible-knn", make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1))),
    ]
    rows = []
    for label, model in models:
        model.fit(x_tr, y_tr)
        train_error = 1.0 - accuracy_score(y_tr, model.predict(x_tr))
        val_error = 1.0 - accuracy_score(y_te, model.predict(x_te))
        variance_proxy = abs(val_error - train_error)
        complexity_cost = 0.01 if label == "linear" else 0.04
        rows.append({
            "label": label,
            "train_error": float(train_error),
            "val_error": float(val_error),
            "variance_proxy": float(variance_proxy),
            "score": float(val_error + variance_proxy + complexity_cost),
        })
    return rows

def run_bias_variance_ladder():
    rows = []
    for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
        candidates = bias_variance_candidates_for_rung(X, y)
        winner = min(candidates, key=lambda item: item["score"])
        rows.append({
            "rung": rung,
            "name": name,
            "n": X.shape[0],
            "d": X.shape[1],
            "classes": len(np.unique(y)),
            "metric": winner["val_error"],
            "gap": winner["variance_proxy"],
            "model": winner["label"],
            "score": winner["score"],
        })
    return rows

def add_intercept(X):
    ones = np.ones((X.shape[0], 1))
    return np.hstack([ones, X])

def train_binary_perceptron(X, y_signed, epochs=60):
    X_aug = add_intercept(X)
    weights = np.zeros(X_aug.shape[1])
    mistakes = []
    for epoch in range(epochs):
        errors = 0
        for xi, yi in zip(X_aug, y_signed):
            margin = yi * float(np.dot(weights, xi))
            if margin <= 0:
                weights = weights + yi * xi
                errors = errors + 1
        mistakes.append(errors)
        if errors == 0:
            break
    return weights, mistakes

def train_ovr_perceptron(X, y, epochs=60):
    classes = np.unique(y)
    weights = []
    histories = []
    for cls in classes:
        y_signed = np.where(y == cls, 1, -1)
        w, hist = train_binary_perceptron(X, y_signed, epochs=epochs)
        weights.append(w)
        histories.append(hist)
    return classes, np.vstack(weights), histories

def predict_ovr_perceptron(classes, weights, X):
    scores = add_intercept(X).dot(weights.T)
    return classes[np.argmax(scores, axis=1)]

def perceptron_predictor(x_tr, y_tr, x_te):
    classes, weights, histories = train_ovr_perceptron(x_tr, y_tr, epochs=60)
    return predict_ovr_perceptron(classes, weights, x_te)

def run_perceptron_ladder():
    rows = []
    for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
        stratify = safe_stratify(y)
        x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=stratify)
        scaler = StandardScaler()
        x_tr_s = scaler.fit_transform(x_tr)
        x_te_s = scaler.transform(x_te)
        classes, weights, histories = train_ovr_perceptron(x_tr_s, y_tr, epochs=60)
        pred = predict_ovr_perceptron(classes, weights, x_te_s)
        rows.append({
            "rung": rung,
            "name": name,
            "n": X.shape[0],
            "d": X.shape[1],
            "classes": len(np.unique(y)),
            "metric": float(accuracy_score(y_te, pred)),
            "history": histories,
        })
    return rows


## The dataset ladder

D1 is inspectable by hand; D5 is a real 30-dimensional breast-cancer classification problem.

In [ ]:

rungs = clf_ladder()
for name, X, y in rungs:
    values, counts = np.unique(y, return_counts=True)
    preview = np.round(X[:3, :min(4, X.shape[1])], 3)
    print(name)
    print("  shape:", X.shape)
    print("  class counts:", dict(zip(values.tolist(), counts.tolist())))
    print("  sample columns:")
    print(preview)


## Run the same method across D1–D5

The metric follows the plan: validation loss and generalization gap for 3.1–3.3, accuracy for 3.4.

In [ ]:

results = run_logistic_ladder()
print("rung | validation_loss | generalization_gap | accuracy | C | score")
for row in results:
    print(f"D{row['rung']} | {row['metric']:.3f} | {row['gap']:.3f} | {row['accuracy']:.3f} | {row['C']} | {row['score']:.3f}")
helper_acc = clf_accuracy(logistic_baseline, rungs[-1][1], rungs[-1][2])
print("D5 logistic_baseline accuracy:", round(helper_acc, 3))


## Results visualization

The closing figure has one panel per rung plus a summary curve over D1–D5.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
flat_axes = axes.ravel()
for ax, (name, X, y), row in zip(flat_axes[:5], rungs, results):
    plot_2d_projection(ax, X, y, f"D{row['rung']} validation loss={row['metric']:.2f}")
flat_axes[5].plot([row["rung"] for row in results], [row["metric"] for row in results], marker="o", label="validation loss")
if "class" != "perceptron":
    flat_axes[5].plot([row["rung"] for row in results], [abs(row["gap"]) for row in results], marker="s", label="gap")
flat_axes[5].set_xlabel("rung")
flat_axes[5].set_ylabel("loss / gap")
flat_axes[5].legend()
fig.tight_layout()
plt.show()


## Pitfall on D5: optimizing the raw term and forgetting the cost

The hardest rung demonstrates why the raw term alone is not the decision rule.

In [ ]:

d5_name, d5_X, d5_y = rungs[-1]
d5_candidates, raw_winner, fixed_winner = logistic_candidates_for_rung(d5_X, d5_y)
print("D5:", d5_name)
print("wrong raw winner C:", raw_winner["C"], "validation loss", round(raw_winner["val_loss"], 3))
print("fixed winner C:", fixed_winner["C"], "validation loss + cost", round(fixed_winner["score"], 3))
print("fixed gap check:", round(fixed_winner["gap"], 3))
assert fixed_winner["score"] <= raw_winner["val_loss"] + raw_winner["cost"] + 1e-9


## Evaluate it + Practice

- Compare the metric with a no-skill baseline or `logistic_baseline`.
- Sanity check: shuffle labels and confirm the score degrades.
- Ablation: turn off the cost or scaling fix and watch the D5 choice or metric change.
- Failure signals: a large validation gap, a scale mismatch, or a raw-only winner.

Practice 1: Change one cost and rerun the D5 selection.

Practice 2: Repeat the D5 split with a different seed and compare the gap.

Practice 3: For skewed classes, add macro-F1 and compare it with accuracy.